In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats

import s3_utils

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 25)
pd.set_option('display.width', 220)

In [2]:
print('Loading two-stage Winogender results and metadata...')

pronoun_df = s3_utils.read_csv('outputs/gpt2-xl/winogender/pronoun_probs.csv')
suffix_df = s3_utils.read_csv('outputs/gpt2-xl/winogender/suffix_probs.csv')
impact_df = s3_utils.read_csv('outputs/gpt2-xl/winogender/accumulated_impact_winogender.csv')
metadata = s3_utils.read_json('data/winogender/winogender_metadata.json')

meta_df = pd.DataFrame(metadata)

pronoun_df = pronoun_df.merge(meta_df, left_on='ID', right_on='id', how='left')
suffix_df = suffix_df.merge(meta_df, left_on='ID', right_on='id', how='left')
impact_df = impact_df.merge(meta_df, left_on='ID', right_on='id', how='left')

print(f'Stage 1 (pronoun DLA) rows: {len(pronoun_df)}, unique IDs: {pronoun_df["ID"].nunique()}')
print(f'Stage 2 (suffix probs) rows: {len(suffix_df)}, unique IDs: {suffix_df["ID"].nunique()}')
print(f'Impact rows: {len(impact_df)}')
print(f'  answer=0 (pronoun=occupation): {meta_df[meta_df["answer"]==0].shape[0]}')
print(f'  answer=1 (pronoun=participant): {meta_df[meta_df["answer"]==1].shape[0]}')

Loading two-stage Winogender results and metadata...


NoSuchKey: An error occurred (NoSuchKey) when calling the GetObject operation: The specified key does not exist.

In [ ]:
def compute_metrics(prob_data):
    """Compute SS, LMS, ICAT from Stage 1 pronoun probability DataFrame."""
    max_layer = prob_data['Layer'].max()
    last_tok_idx = prob_data.groupby(['ID', 'Type', 'Layer'])['Token_Position'].idxmax()
    final = prob_data.loc[last_tok_idx]
    final = final[final['Layer'] == max_layer]
    pivot = final.pivot(index='ID', columns='Type', values='Layer_Accumulated_Prob').fillna(0)

    n_total = len(pivot)
    related = 0
    n_stereo = 0
    n_anti = 0

    has_all_three = {'stereotype', 'anti-stereotype', 'unrelated'} <= set(pivot.columns)
    has_stereo_anti = {'stereotype', 'anti-stereotype'} <= set(pivot.columns)

    if has_all_three:
        related += int((pivot['stereotype'] > pivot['unrelated']).sum())
        related += int((pivot['anti-stereotype'] > pivot['unrelated']).sum())
    if has_stereo_anti:
        n_stereo = int((pivot['stereotype'] > pivot['anti-stereotype']).sum())
        n_anti = int((pivot['anti-stereotype'] > pivot['stereotype']).sum())

    lms = (related / (2 * n_total) * 100) if (n_total > 0 and has_all_three) else 0.0
    denom = n_stereo + n_anti
    ss = (n_stereo / denom * 100) if denom > 0 else 50.0
    icat = lms * (min(ss, 100.0 - ss) / 50.0)

    return ss, lms, icat

In [ ]:
pronoun_occ = pronoun_df[pronoun_df['answer'] == 0].copy()
pronoun_part = pronoun_df[pronoun_df['answer'] == 1].copy()

ss_all, lms_all, icat_all = compute_metrics(pronoun_df)
ss_occ, lms_occ, icat_occ = compute_metrics(pronoun_occ)
ss_part, lms_part, icat_part = compute_metrics(pronoun_part)

summary = pd.DataFrame([
    {'Subset': 'All (120 templates)', 'SS': ss_all, 'LMS': lms_all, 'ICAT': icat_all},
    {'Subset': 'answer=0 (pronoun=occupation)', 'SS': ss_occ, 'LMS': lms_occ, 'ICAT': icat_occ},
    {'Subset': 'answer=1 (pronoun=participant)', 'SS': ss_part, 'LMS': lms_part, 'ICAT': icat_part},
])

print('Stage 1: Pronoun Prediction Metrics (GPT-2 XL Baseline)')
print('  SS > 50 means the model prefers the stereotype pronoun at the prediction point.')
print('  answer=0 = primary bias measure | answer=1 = control group\n')
display(summary.style.format({'SS': '{:.2f}', 'LMS': '{:.2f}', 'ICAT': '{:.2f}'}).hide(axis='index'))

In [ ]:
# ── Stage 1: pronoun preference (last-layer probability) ─────────────
max_layer = pronoun_df['Layer'].max()
last_tok = pronoun_df.groupby(['ID', 'Type', 'Layer'])['Token_Position'].idxmax()
final_pronoun = pronoun_df.loc[last_tok]
final_pronoun = final_pronoun[final_pronoun['Layer'] == max_layer]

pref_pivot = final_pronoun.pivot(index='ID', columns='Type',
                                  values='Layer_Accumulated_Prob').fillna(0)
pref_pivot['pronoun_preference'] = pref_pivot.idxmax(axis=1)
pref_pivot['P_stereo'] = pref_pivot.get('stereotype', 0)
pref_pivot['P_anti'] = pref_pivot.get('anti-stereotype', 0)
pref_pivot['P_neutral'] = pref_pivot.get('unrelated', 0)

# ── Stage 2: coreference resolution (suffix log-prob) ────────────────
coref_pivot = suffix_df.pivot(index='ID', columns='Type',
                               values='Suffix_Log_Prob').fillna(-1e9)
coref_pivot.columns = [f'suffix_{c}' for c in coref_pivot.columns]
coref_pivot['coreference_choice'] = coref_pivot.idxmax(axis=1).str.replace('suffix_', '')

# ── Combine ──────────────────────────────────────────────────────────
combined = pref_pivot[['pronoun_preference', 'P_stereo', 'P_anti', 'P_neutral']].merge(
    coref_pivot, left_index=True, right_index=True
).merge(meta_df, left_index=True, right_on='id')

combined['is_stereotypical'] = (
    (combined['pronoun_preference'] == 'stereotype') &
    (combined['coreference_choice'] == 'stereotype')
)
combined['is_anti_stereotypical'] = (
    (combined['pronoun_preference'] == 'anti-stereotype') &
    (combined['coreference_choice'] == 'anti-stereotype')
)

print('Two-stage bias determination (per template):\n')
display(combined[['id', 'occupation', 'answer', 'bls_pct_female',
                   'pronoun_preference', 'coreference_choice',
                   'is_stereotypical', 'is_anti_stereotypical',
                   'P_stereo', 'P_anti', 'P_neutral']].sort_values(
    ['occupation', 'answer']))

# Per-answer-type summary
for ans, label in [(0, 'answer=0 (pronoun=occupation)'), (1, 'answer=1 (pronoun=participant)')]:
    sub = combined[combined['answer'] == ans]
    n = len(sub)
    stereo_rate = sub['is_stereotypical'].sum() / n * 100
    anti_rate = sub['is_anti_stereotypical'].sum() / n * 100
    print(f'\n{label} (n={n}):')
    print(f'  Stereotypical rate:      {stereo_rate:.1f}%')
    print(f'  Anti-stereotypical rate:  {anti_rate:.1f}%')

In [ ]:
# Per-occupation two-stage bias score (answer=0 only)
occ_combined = combined[combined['answer'] == 0].copy()

# Bias score: P(stereo suffix) - P(anti-stereo suffix), normalized by coreference
occ_combined['coref_bias'] = (
    occ_combined['suffix_stereotype'] - occ_combined['suffix_anti-stereotype']
)
occ_combined['pronoun_bias'] = occ_combined['P_stereo'] - occ_combined['P_anti']

# Combined score: product of pronoun preference direction and coreference direction
# Positive = both stages agree on stereotype direction
occ_combined['combined_bias'] = np.sign(occ_combined['pronoun_bias']) * np.abs(occ_combined['coref_bias'])

occ_combined['gender_majority'] = occ_combined['bls_pct_female'].apply(
    lambda x: 'Female-dominated (>50%)' if x > 50 else 'Male-dominated (<=50%)'
)
occ_sorted = occ_combined.sort_values('combined_bias')

fig, ax = plt.subplots(figsize=(10, 16))
colors = occ_sorted['gender_majority'].map({
    'Female-dominated (>50%)': '#e74c3c',
    'Male-dominated (<=50%)': '#3498db'
})
ax.barh(occ_sorted['occupation'], occ_sorted['combined_bias'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Combined Bias Score (pronoun pref. direction x coreference strength)', fontsize=11)
ax.set_title('Per-Occupation Two-Stage Bias Score\n(answer=0, GPT-2 XL Baseline)',
             fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Female-dominated (>50%)'),
    Patch(facecolor='#3498db', label='Male-dominated (<=50%)')
]
ax.legend(handles=legend_elements, loc='lower right')
sns.despine(ax=ax)
plt.tight_layout()
s3_utils.save_plot(fig, 'outputs/gpt2-xl/winogender/plots/per_occupation_two_stage_bias.pdf')
plt.show()

In [ ]:
# BLS correlation using coreference-based bias score
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Stage 1 pronoun bias vs BLS
ax = axes[0]
ax.scatter(occ_combined['bls_pct_female'], occ_combined['pronoun_bias'],
           alpha=0.7, s=60, edgecolor='black', linewidth=0.5)
for _, row in occ_combined.iterrows():
    ax.annotate(row['occupation'], (row['bls_pct_female'], row['pronoun_bias']),
                fontsize=7, alpha=0.7, ha='center', va='bottom')

x = occ_combined['bls_pct_female'].values
y = occ_combined['pronoun_bias'].values
sl, ic, r_val, p_val, _ = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, sl * x_line + ic, 'r--', linewidth=2,
        label=f'r={r_val:.3f}, p={p_val:.2e}')
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.set_xlabel('BLS % Female', fontsize=12)
ax.set_ylabel('P(stereo) - P(anti-stereo)', fontsize=12)
ax.set_title('Stage 1: Pronoun Preference vs BLS', fontweight='bold')
ax.legend(fontsize=10)
sns.despine(ax=ax)

# Right: Stage 2 coreference bias vs BLS
ax = axes[1]
ax.scatter(occ_combined['bls_pct_female'], occ_combined['coref_bias'],
           alpha=0.7, s=60, edgecolor='black', linewidth=0.5)
for _, row in occ_combined.iterrows():
    ax.annotate(row['occupation'], (row['bls_pct_female'], row['coref_bias']),
                fontsize=7, alpha=0.7, ha='center', va='bottom')

y2 = occ_combined['coref_bias'].values
sl2, ic2, r_val2, p_val2, _ = stats.linregress(x, y2)
ax.plot(x_line, sl2 * x_line + ic2, 'r--', linewidth=2,
        label=f'r={r_val2:.3f}, p={p_val2:.2e}')
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax.set_xlabel('BLS % Female', fontsize=12)
ax.set_ylabel('Suffix logP(stereo) - Suffix logP(anti-stereo)', fontsize=12)
ax.set_title('Stage 2: Coreference Bias vs BLS', fontweight='bold')
ax.legend(fontsize=10)
sns.despine(ax=ax)

plt.suptitle('Correlation with Real-World Gender Statistics (answer=0, GPT-2 XL)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
s3_utils.save_plot(fig, 'outputs/gpt2-xl/winogender/plots/bls_correlation_two_stage.pdf')
plt.show()

print(f'Stage 1 Pronoun Preference: r={r_val:.4f}, p={p_val:.4e}')
print(f'Stage 2 Coreference Bias:   r={r_val2:.4f}, p={p_val2:.4e}')

In [ ]:
def generate_grouped_analysis(sub_df, scenario_title, all_probs_df, winner_type):
    """3-panel impact plot: MLP impact, Head sum, Head heatmap."""
    if len(sub_df) == 0:
        return

    relevant_ids = sub_df['ID'].unique()
    current_probs = all_probs_df[all_probs_df['ID'].isin(relevant_ids)].copy()

    type_map = {'stereotype': 'Stereotype', 'anti-stereotype': 'Anti-Stereotype', 'unrelated': 'Unrelated'}
    current_probs['Type'] = current_probs['Type'].map(type_map).fillna(current_probs['Type'])
    plot_df = sub_df.copy()
    plot_df['Type'] = plot_df['Type'].map(type_map).fillna(plot_df['Type'])
    winner_label = type_map.get(winner_type, winner_type)

    mlp_df = plot_df[plot_df['Component'] == 'MLP']
    mlp_summary = mlp_df.groupby(['Layer', 'Type'])['Accumulated_Impact'].mean().reset_index()

    prob_summary = current_probs.groupby(['Layer', 'Type'])['Layer_Accumulated_Prob'].mean().reset_index()
    prob_pivot = prob_summary.pivot(index='Layer', columns='Type', values='Layer_Accumulated_Prob').fillna(0)
    prob_summary_winner = pd.DataFrame({'Layer': prob_pivot.index})

    if winner_label == 'Stereotype':
        prob_summary_winner['Margin'] = prob_pivot.get('Stereotype', 0) - prob_pivot.get('Anti-Stereotype', 0)
        y_label = 'Margin: P(Stereo) - P(Anti)'
    elif winner_label == 'Anti-Stereotype':
        prob_summary_winner['Margin'] = prob_pivot.get('Anti-Stereotype', 0) - prob_pivot.get('Stereotype', 0)
        y_label = 'Margin: P(Anti) - P(Stereo)'
    else:
        prob_summary_winner['Margin'] = prob_pivot.get(winner_label, 0)
        y_label = f'Prob ({winner_label})'

    head_df = plot_df[plot_df['Component'].str.startswith('Head')].copy()
    head_df['Head_ID'] = head_df['Component'].str.replace('Head_', '').astype(int)
    heatmap_data = head_df[head_df['Type'] == winner_label]
    head_matrix = heatmap_data.groupby(['Head_ID', 'Layer'])['Accumulated_Impact'].mean().unstack()

    head_layer_sum = head_df.groupby(['ID', 'Layer', 'Type'])['Accumulated_Impact'].sum().reset_index()
    head_sum_summary = head_layer_sum.groupby(['Layer', 'Type'])['Accumulated_Impact'].mean().reset_index()

    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 0.9], hspace=0.3, wspace=0.25)
    ax_mlp = fig.add_subplot(gs[0, 0])
    ax_sum = fig.add_subplot(gs[0, 1])
    ax_heat = fig.add_subplot(gs[1, :])

    ax_mlp_twin = ax_mlp.twinx()
    sns.barplot(data=prob_summary_winner, x='Layer', y='Margin',
                color='lightgray', alpha=0.5, ax=ax_mlp_twin, errorbar=None)
    ax_mlp_twin.set_ylabel(y_label, color='gray', fontweight='bold')
    ax_mlp_twin.grid(False)
    ax_mlp_twin.set_xticks([])

    sns.lineplot(data=mlp_summary, x='Layer', y='Accumulated_Impact', hue='Type',
                 marker='o', linewidth=2.5, ax=ax_mlp, palette='colorblind')
    ax_mlp.set_title('MLP Impact & Probability Margin', fontweight='bold')
    ax_mlp.set_ylabel('Cumulative Logit Contribution')
    ax_mlp.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_mlp.legend(title='', loc='upper left', frameon=False)
    sns.despine(ax=ax_mlp, right=False)

    sns.lineplot(data=head_sum_summary, x='Layer', y='Accumulated_Impact', hue='Type',
                 marker='s', linewidth=2.5, ax=ax_sum, palette='colorblind')
    ax_sum.set_title('Total Attention Block Impact (Sum of Heads)', fontweight='bold')
    ax_sum.set_ylabel('Cumulative Logit Contribution')
    ax_sum.xaxis.set_major_locator(ticker.MultipleLocator(5))
    ax_sum.legend(title='', loc='upper left', frameon=False)
    sns.despine(ax=ax_sum)

    if head_matrix is not None and not head_matrix.empty:
        sns.heatmap(head_matrix, cmap='RdBu_r', center=0, ax=ax_heat,
                    xticklabels=5, yticklabels=2, cbar_kws={'label': 'Accumulated Impact'})
        ax_heat.set_title(f'Per-Head Impact Heatmap ({winner_label})', fontweight='bold')
        ax_heat.set_xlabel('Layer')
        ax_heat.set_ylabel('Head')

    plt.suptitle(scenario_title, fontsize=18, fontweight='bold', y=0.96)

    safe_title = scenario_title.replace(' ', '_').replace('(', '').replace(')', '').replace(':', '').replace('%', '').replace('\n', '_')
    s3_utils.save_plot(fig, f'outputs/gpt2-xl/winogender/plots/{safe_title}_impact_analysis.pdf')
    plt.show()
    plt.close(fig)


# Top-5 most biased occupations by |combined_bias|
top5 = occ_combined.reindex(
    occ_combined['combined_bias'].abs().sort_values(ascending=False).index
).head(5)

for _, row in top5.iterrows():
    occ = row['occupation']
    occ_ids = meta_df[(meta_df['answer'] == 0) & (meta_df['occupation'] == occ)]['id'].values
    sub_impact = impact_df[impact_df['ID'].isin(occ_ids)]
    sub_probs = pronoun_df[pronoun_df['ID'].isin(occ_ids)]

    direction = 'stereotype' if row['combined_bias'] > 0 else 'anti-stereotype'
    title = (f'{occ.title()} (bias={row["combined_bias"]:.4f}, '
             f'BLS {row["bls_pct_female"]:.0f}% female)')
    generate_grouped_analysis(sub_impact, title, sub_probs, direction)

In [ ]:
# Optional: compare baseline vs fine-tuned models on Winogender
ft_keys = s3_utils.list_keys('outputs/gpt2-xl/winogender/finetuned/')
prefix_s3 = s3_utils.s3_key('outputs/gpt2-xl/winogender/finetuned/')

ft_run_ids = set()
for k in ft_keys:
    parts = k[len(prefix_s3):].split('/')
    if len(parts) >= 1 and parts[0]:
        ft_run_ids.add(parts[0])

ft_run_ids = sorted(ft_run_ids)
print(f'Found {len(ft_run_ids)} fine-tuned Winogender evaluations.')

if ft_run_ids:
    comparison_rows = [{
        'run_id': 'baseline',
        'SS_all': ss_all, 'LMS_all': lms_all, 'ICAT_all': icat_all,
        'SS_occ': ss_occ, 'LMS_occ': lms_occ, 'ICAT_occ': icat_occ,
    }]

    for rid in ft_run_ids:
        try:
            ft_prob = s3_utils.read_csv(
                f'outputs/gpt2-xl/winogender/finetuned/{rid}/pronoun_probs.csv')
            ft_prob = ft_prob.merge(meta_df, left_on='ID', right_on='id', how='left')

            s_all, l_all, i_all = compute_metrics(ft_prob)
            s_occ, l_occ, i_occ = compute_metrics(ft_prob[ft_prob['answer'] == 0])

            comparison_rows.append({
                'run_id': rid,
                'SS_all': s_all, 'LMS_all': l_all, 'ICAT_all': i_all,
                'SS_occ': s_occ, 'LMS_occ': l_occ, 'ICAT_occ': i_occ,
            })
        except Exception as e:
            print(f'  Skipping {rid}: {e}')

    comp_df = pd.DataFrame(comparison_rows)
    print('\nBaseline vs Fine-tuned (Winogender):')
    display(comp_df.style.format({
        c: '{:.2f}' for c in comp_df.columns if c != 'run_id'
    }).hide(axis='index'))
else:
    print('No fine-tuned Winogender evaluations found.')
    print('Run: python winogender_bias_search.py --run_id <id>')